# NPU Lab · SDXL Turbo
Select a GPU runtime. Run the setup and model cells, then try the in-notebook example. The model cache defaults to the temporary Colab runtime disk.

Run the optional tunnel cell only on a runtime whose terms allow using the NPU Lab dashboard as an external interface. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html). The tunnel needs no Google Cloud project or provider API key. Its pairing token is generated by this notebook.

In [ ]:
%pip -q install diffusers transformers accelerate safetensors flask

In [ ]:
import base64, io, os, secrets, shutil, threading, time, uuid
import numpy as np
import torch
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
from PIL import Image

MODEL = "stabilityai/sdxl-turbo"
CACHE = "/content/npu-lab-model-cache"
# For an optional Drive cache, mount Drive yourself and point CACHE at a dedicated folder.
# Check your account's available storage first. Keep the default for temporary storage.
os.makedirs(CACHE, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before loading SDXL Turbo.")
if CACHE.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Mount Drive before choosing a Drive cache.")
if shutil.disk_usage(CACHE).free < 8 * 1024**3:
    raise RuntimeError("Allow at least 8 GiB free for model downloads and temporary files.")
started = time.monotonic()
text_pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
    cache_dir=CACHE, token=False,
).to("cuda")
reference_pipe = AutoPipelineForImage2Image.from_pipe(text_pipe)
print(f"Loaded {MODEL} in {time.monotonic() - started:.1f}s.")

def generate(settings, callback=None):
    prompt = settings.get("prompt", "").strip()
    steps = settings.get("steps", 4)
    seed = settings.get("seed", 42)
    reference = settings.get("reference")
    if len(prompt) > 1000 or (not prompt and not reference):
        raise ValueError("Enter a prompt or provide a reference image.")
    if type(steps) is not int or not 1 <= steps <= 8:
        raise ValueError("Choose 1 to 8 steps.")
    if type(seed) is not int or not 0 <= seed <= 2147483647:
        raise ValueError("Invalid seed.")
    options = dict(prompt=prompt, num_inference_steps=steps, guidance_scale=0.0,
                   generator=torch.Generator(device="cuda").manual_seed(seed),
                   callback_on_step_end=callback)
    prepared = None
    if reference:
        strength = settings.get("strength", 0.5)
        if type(strength) not in (int, float) or not 0.25 <= strength <= 1 or steps * strength < 1:
            raise ValueError("Use strength 0.25–1 and enough steps for one denoising step.")
        with Image.open(io.BytesIO(base64.b64decode(reference, validate=True))) as source:
            if source.size != (256, 256):
                raise ValueError("Reference input must be 256 × 256 pixels.")
            prepared = source.convert("RGB").resize((512, 512), Image.Resampling.LANCZOS)
        options.update(image=prepared, strength=strength)
        pipe = reference_pipe
    else:
        options.update(width=512, height=512)
        pipe = text_pipe
    started = time.monotonic()
    output = pipe(**options).images[0]
    result = dict(device="Colab", model=MODEL, setupSeconds=0, runSeconds=time.monotonic()-started, seed=seed)
    if prepared is not None:
        a = np.asarray(output.resize((256, 256), Image.Resampling.LANCZOS), dtype=np.float32)
        b = np.asarray(prepared.resize((256, 256), Image.Resampling.LANCZOS), dtype=np.float32)
        result["rgbDifference"] = float(np.abs(a-b).mean()/255*100)
    return output, result

In [ ]:
# Run directly in the notebook.
from IPython.display import display
image, info = generate({"prompt": "Watercolor fox in an autumn forest", "seed": 42, "steps": 4})
display(image)
print(info)

## Optional dashboard bridge
Use only where the runtime allows external interfaces. Keep this notebook open. Copy the temporary URL and pairing token into **Image studio → Connect Colab**. Stop this cell to close the tunnel. Keep the pairing token private.

In [ ]:
import hmac, re, subprocess
from flask import Flask, abort, jsonify, request
from werkzeug.serving import make_server

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 262_144
token = secrets.token_urlsafe(32)
lock = threading.Lock()
jobs = {}

@app.before_request
def authenticate():
    received = request.headers.get("Authorization", "").encode()
    if not hmac.compare_digest(received, ("Bearer " + token).encode()):
        abort(401)

def worker(job, settings):
    def check_lease(_pipe, _step, _time, values):
        if time.monotonic() > job["expires"]:
            raise RuntimeError("Colab run stopped because the dashboard disconnected.")
        return values
    try:
        image, result = generate(settings, check_lease)
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        result["png"] = base64.b64encode(buffer.getvalue()).decode("ascii")
        with lock:
            job.update(status="done", result=result)
    except Exception as error:
        with lock:
            job.update(status="error", error=str(error)[:1000])

@app.post("/jobs")
def submit():
    settings = request.get_json()
    if not isinstance(settings, dict):
        abort(400)
    with lock:
        if any(job["status"] == "running" for job in jobs.values()):
            abort(409)
        jobs.clear()
        key = uuid.uuid4().hex
        job = {"status": "running", "expires": time.monotonic() + 25}
        jobs[key] = job
        threading.Thread(target=worker, args=(job, settings), daemon=True).start()
    return jsonify(id=key), 202

@app.get("/jobs/<key>")
def poll(key):
    with lock:
        job = jobs.get(key)
        if job is None:
            abort(404)
        if time.monotonic() < job["expires"]:
            job["expires"] = time.monotonic() + 25
        return jsonify({key: value for key, value in job.items() if key != "expires"})

binary = "/tmp/npu-lab-cloudflared"
if not os.path.isfile(binary):
    subprocess.run(["curl", "--fail", "--location", "--output", binary,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(binary, 0o700)

server = make_server("127.0.0.1", 7860, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
tunnel = None
try:
    tunnel = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in tunnel.stdout:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("Tunnel URL:", match.group(0), flush=True)
            print("Pairing token:", token, flush=True)
            break
    tunnel.wait()
finally:
    if tunnel is not None:
        tunnel.terminate()
    server.shutdown()
    server.server_close()